In [1]:
using Pkg
Pkg.activate(".")

Pkg.resolve()
Pkg.instantiate()
Pkg.precompile()


  Activating project at `~/codes/Julia/MyPack/SpectralMM`
     Project No packages added to or removed from `~/codes/Julia/MyPack/SpectralMM/Project.toml`
    Manifest No packages added to or removed from `~/codes/Julia/MyPack/SpectralMM/Manifest.toml`


In [2]:
using SpectralMM
using GLM
using Distributions
using Random
using LinearAlgebra

[ Info: Precompiling SpectralMM [3ead10a5-a636-4c1d-b4eb-bf43a9e33d59] (cache misses: wrong dep version loaded (2), mismatched flags (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [3]:
using Random, SparseArrays, BenchmarkTools, SpectralMM

const T = Float64
const SEED = 3124

function sparse_design(rng, n, p; nnz_per_row=20)
    m = n * nnz_per_row
    I = repeat(1:n, inner=nnz_per_row)
    J = rand(rng, 1:p, m)
    V = randn(rng, T, m)
    sparse(I, J, V, n, p)
end

for p in [500, 1000, 2000, 5000]
    n = 2p
    rng = MersenneTwister(SEED)

    X = sparse_design(rng, n, p)
    βtrue = randn(rng, T, p) ./ sqrt(T(p))
    y, _, _ = SpectralMM.simulate_y_from_beta(
        X, βtrue, SpectralMM.GaussianIdentity();
        seed=SEED, beta0=0.5,
    )

    Xfit = hcat(sparse(ones(T, n)), X)
    β0 = SpectralMM._init_beta(
        Xfit, y, SpectralMM.GaussianIdentity(), nothing
    )

    f_jacobi = () -> SpectralMM.irls_krylov(
        Xfit, y;
        family=SpectralMM.GaussianIdentity(),
        beta0=β0,
        solver=:cgls,
        ridge=0.0,
        maxiter=1000,
        gtol=1e-6,
        relgtol=1e-8,
        outer_nesterov=true,
        preconditioner=:jacobi,
        verbose=false,
    )

    f_none = () -> SpectralMM.irls_krylov(
        Xfit, y;
        family=SpectralMM.GaussianIdentity(),
        beta0=β0,
        solver=:cgls,
        ridge=0.0,
        maxiter=1000,
        gtol=1e-6,
        relgtol=1e-8,
        outer_nesterov=true,
        preconditioner=:none,
        verbose=false,
    )

    f_jacobi(); f_none()

    tj = @belapsed $f_jacobi()
    tn = @belapsed $f_none()

    println(
        "p=$p, nnz=$(nnz(X)), ",
        "Jacobi=$(round(1000tj, digits=3)) ms, ",
        "None=$(round(1000tn, digits=3)) ms"
    )
end

p=500, nnz=19597, Jacobi=0.932 ms, None=1.147 ms
p=1000, nnz=39642, Jacobi=1.971 ms, None=2.551 ms
p=2000, nnz=79644, Jacobi=4.11 ms, None=5.426 ms
p=5000, nnz=199657, Jacobi=12.173 ms, None=16.792 ms


In [3]:
using Random
using DataFrames
using Distributions
using GLM
using StatsModels
using SpectralMM

rng = MersenneTwister(3124)

n = 1000
p = 5

X = randn(rng, n, p)
beta = [0.5, 1.0, -1.0, 0.5, 0.0, -0.5]

Xfit = hcat(ones(n), X)
eta = Xfit * beta
prob = 1.0 ./ (1.0 .+ exp.(-eta))
y = Float64.([rand(rng) < pi for pi in prob])

1000-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.0
 1.0
 0.0
 1.0
 0.0
 0.0
 0.0
 1.0
 0.0
 1.0
 ⋮
 0.0
 0.0
 1.0
 1.0
 1.0
 0.0
 1.0
 0.0
 0.0
 1.0
 0.0
 1.0

In [11]:
model = SpectralMM.glm(
    Xfit,
    y,
    Bernoulli(),
    LogitLink(),
    verbose=true,
)



SpectralMM  Solver: PCG   Rank: 5

Iter          Loss     RelGrad  Inner    InnerRes      EigRes    Step  Spectral
   1    5.2796e+02    9.96e-01      1    2.44e-10    4.91e-16    1.00  init+correct
   2    5.1953e+02    1.77e-01      1    6.10e-02    9.03e-16    1.00  correct
   3    5.1950e+02    9.29e-03      1    1.91e-09           -    1.00  reuse
   4    5.1950e+02    9.29e-05      1    8.24e-04           -    1.00  reuse
   5    5.1950e+02    7.69e-08      1    2.24e-03           -    1.00  reuse

Converged after 5 iterations
Final loss:      5.195000e+02
Relative grad.:  1.720e-10


SpectralGLM{Float64, Bernoulli{Float64}, LogitLink, MMResult{Float64}, Matrix{Float64}, Vector{Float64}, Nothing}([0.6283974269785263, 0.9224885084242418, -0.9244066103907828, 0.3788812365156265, 0.0015256608460288551, -0.4866150128440027], Bernoulli{Float64}(p=0.5), LogitLink(), MMResult{Float64}([0.6283974269785263, 0.9224885084242418, -0.9244066103907828, 0.3788812365156265, 0.0015256608460288551, -0.4866150128440027], [670.503749160752, 527.9607043091753, 519.5274988536926, 519.4999654240055, 519.4999625473023, 519.4999625473007], [245.560897657322, 43.73233734562533, 2.289757947570161, 0.022894583545857505, 1.896219782536722e-5, 4.2404727367368086e-8], [0.9959442068490932, 0.17736931427953306, 0.00928678460098948, 9.285569513815248e-5, 7.690675206626428e-8, 1.719848028226418e-10], [3.620110680820423e-16, 4.908625489853541e-16, 9.033000772635684e-16, 9.033000772635684e-16, 9.033000772635684e-16], [1, 1, 1, 1, 1], [2.438481385451348e-10, 0.06098956230603574, 1.907427859640443e-9, 0.

In [5]:
coef(model)

6-element Vector{Float64}:
  0.6283974269785263
  0.9224885084242418
 -0.9244066103907828
  0.3788812365156265
  0.0015256608460288551
 -0.4866150128440027

In [9]:
data = DataFrame(
    y = y,
    x1 = X[:, 1],
    x2 = X[:, 2],
    x3 = X[:, 3],
    x4 = X[:, 4],
    x5 = X[:, 5],
)

model_formula = SpectralMM.glm(
    @formula(y ~ x1 + x2 + x3 + x4 + x5),
    data,
    Bernoulli(),
    LogitLink(),
    verbose=true,
)

# model_formula


SpectralMM  Solver: PCG   Rank: 5

Iter          Loss     RelGrad  Inner    InnerRes      EigRes    Step  Spectral
   1    5.2796e+02    9.96e-01      1    2.44e-10    4.91e-16    1.00  init+correct
   2    5.1953e+02    1.77e-01      1    6.10e-02    9.03e-16    1.00  correct
   3    5.1950e+02    9.29e-03      1    1.91e-09           -    1.00  reuse
   4    5.1950e+02    9.29e-05      1    8.24e-04           -    1.00  reuse
   5    5.1950e+02    7.69e-08      1    2.24e-03           -    1.00  reuse

Converged after 5 iterations
Final loss:      5.195000e+02
Relative grad.:  1.720e-10


SpectralGLM{Float64, Bernoulli{Float64}, LogitLink, MMResult{Float64}, Matrix{Float64}, Vector{Float64}, FormulaTerm{ContinuousTerm{Float64}, MatrixTerm{Tuple{InterceptTerm{true}, Vararg{ContinuousTerm{Float64}, 5}}}}}([0.6283974269785263, 0.9224885084242418, -0.9244066103907828, 0.3788812365156265, 0.0015256608460288551, -0.4866150128440027], Bernoulli{Float64}(p=0.5), LogitLink(), MMResult{Float64}([0.6283974269785263, 0.9224885084242418, -0.9244066103907828, 0.3788812365156265, 0.0015256608460288551, -0.4866150128440027], [670.503749160752, 527.9607043091753, 519.5274988536926, 519.4999654240055, 519.4999625473023, 519.4999625473007], [245.560897657322, 43.73233734562533, 2.289757947570161, 0.022894583545857505, 1.896219782536722e-5, 4.2404727367368086e-8], [0.9959442068490932, 0.17736931427953306, 0.00928678460098948, 9.285569513815248e-5, 7.690675206626428e-8, 1.719848028226418e-10], [3.620110680820423e-16, 4.908625489853541e-16, 9.033000772635684e-16, 9.033000772635684e-16, 9.033

In [7]:
coef(model_formula)

6-element Vector{Float64}:
  0.6283974269785263
  0.9224885084242418
 -0.9244066103907828
  0.3788812365156265
  0.0015256608460288551
 -0.4866150128440027